In [ ]:
import os
import boto3
import csv
import io
import asyncio
from dotenv import load_dotenv
from agents import Agent, Runner, WebSearchTool
from pydantic import BaseModel
from typing import Literal

load_dotenv()

BUCKET = "cultivate-mapping-data"
CITYLIST_KEY = "raw/exploration_data/2026_data/04_SHARECITY100/CityList.csv"
INPUT_KEY = "raw/exploration_data/2026_data/04_SHARECITY100/dead_link_report.csv"
OUTPUT_KEY = "raw/exploration_data/2026_data/04_SHARECITY100/llm_classification_web_search.csv"

MODEL = os.environ.get("OPENAI_MODEL", "gpt-5-nano")

s3 = boto3.client("s3", region_name="eu-north-1")

In [ ]:
# load city → country mapping from CityList.csv
body = s3.get_object(Bucket=BUCKET, Key=CITYLIST_KEY)["Body"].read().decode("utf-8-sig")
reader = csv.DictReader(io.StringIO(body))
city_country = {row["City"]: row["Country"] for row in reader if row.get("City")}
print(f"Loaded {len(city_country)} city-country mappings")
list(city_country.items())[:3]

In [ ]:
# load alive URLs from dead_link_report
body = s3.get_object(Bucket=BUCKET, Key=INPUT_KEY)["Body"].read().decode("utf-8")
reader = csv.DictReader(io.StringIO(body))
alive_urls = [(row["city"], row["name"], row["url"]) for row in reader if row["alive"] == "True"]
print(f"Alive URLs to classify: {len(alive_urls)}")

In [ ]:
# resume from checkpoint if exists
done_urls = set()
try:
    body = s3.get_object(Bucket=BUCKET, Key=OUTPUT_KEY)["Body"].read().decode("utf-8")
    reader = csv.DictReader(io.StringIO(body))
    done_urls = {row["url"] for row in reader}
    print(f"Found checkpoint: {len(done_urls)} URLs already classified")
except s3.exceptions.NoSuchKey:
    print("No checkpoint found, starting fresh")

remaining = [(city, name, url) for city, name, url in alive_urls if url not in done_urls]
print(f"Remaining: {len(remaining)} URLs")

In [ ]:
class Classification(BaseModel):
    valid: bool
    confidence: Literal["low", "medium", "high"]


def build_instructions(city: str, country: str) -> str:
    return f"""
Task: Classify whether the given URL represents a valid Food Sharing Initiative (FSI) located in {city}, {country}.

Method (Strict Evidence-Based Evaluation):
1. Retrieve website content using web_search:
   - First, fetch the provided URL and extract its content
   - Find all anchor tags/links on the page that belong to the same domain
   - Follow links that appear to lead to informational pages such as:
     * about, about-us, who-we-are, our-story, mission, vision
   - Retrieve and analyze content from these relevant internal pages
   - Ignore external links, social media links, and unrelated pages

2. Extract only explicit textual evidence from retrieved pages.
   Treat all claims as unverified unless directly supported by extracted text.

3. Do not infer or assume intentions based on tone, imagery, branding, or general community-oriented language.

4. Before producing output, internally evaluate (do not reveal this reasoning):
   - relevant text extracted from each page
   - each VALID criterion as Confirmed, Contradicted, or Not found
   - each INVALID trigger
   - final decision based strictly on explicit evidence

5. If any required page returns insufficient text, or the website is blank, broken, or inaccessible, classify as INVALID with low confidence.

VALID FSI — All Six Conditions Must Be Explicitly Confirmed:

1. Direct representation:
   The website is owned by or officially represents the initiative, not a media site, directory, listing, or article.

2. Explicit food-sharing purpose:
   The site clearly states that the initiative performs food redistribution or communal food sharing, such as:
   - food rescue
   - free or pay-what-you-can meals
   - community kitchens
   - shared gardens where harvest is distributed or collectively available
   - seed/produce swaps
   - non-commercial food cooperatives
   General mentions of sustainability, community, or ecology are insufficient without explicit food-sharing activities.

3. Active food-sharing operations:
   Evidence shows ongoing or recurring food distribution or communal food-sharing activities (not a one-time event).

4. Non-commercial:
   The initiative's primary purpose is community food access, not sales or profit-making.

5. Location match:
   The initiative explicitly states an address or operational activity in {city}, {country}.
   Nearby cities, regions, or generic national presence do not qualify.

6. Evidence of continuity:
   Clear indication of recurring or ongoing programs (events, schedules, regular services).

If any required condition is "Not found", classify as INVALID.

INVALID FSI — Any One of These Is Sufficient:

- News, media, editorial, or blog content describing an initiative.
- Government or municipal pages listing external programs without operating them.
- Crowdfunding or campaign-only pages (GoFundMe, Kickstarter, etc.).
- Social media profiles without a verifiable organizational website.
- Restaurants, cafes, or food-delivery or food-retail businesses.
- Schools or cultural institutions hosting only one-off food events.
- Gardening, farming, ecology, or sustainability groups without explicit food sharing or redistribution.
- Multi-city or international networks without confirmed operations in {city}, {country}.
- Any page with insufficient evidence, ambiguous purpose, or inaccessible/empty content.

Edge Cases:

- Community centers, libraries, churches: Valid only if food sharing is a core, ongoing activity explicitly stated.
- Gardening groups: Valid only if the harvest is shared or redistributed, not merely grown.
- Coalitions or networks: Valid only if they coordinate actual food-sharing activity, not just advocacy or education.

Confidence Scoring:

- High: All required evidence is explicit, consistent, and unambiguous.
- Medium: Most evidence is explicit, but one secondary attribute (e.g., frequency or non-commercial nature) is implied.
- Low: Missing or ambiguous evidence; unclear location; partial page retrieval; or overall uncertainty.
"""


def make_agent(city: str, country: str) -> Agent:
    return Agent(
        name="FSI Classifier",
        model=MODEL,
        instructions=build_instructions(city, country),
        tools=[WebSearchTool()],
        output_type=Classification,
    )


async def classify(city: str, country: str, url: str) -> Classification:
    agent = make_agent(city, country)
    result = await Runner.run(agent, f"Classify this url: {url}")
    return result.final_output

In [ ]:
# test with first 3 URLs
test_batch = remaining[:3]
for city, name, url in test_batch:
    country = city_country.get(city, "")
    try:
        result = await classify(city, country, url)
        print(f"{city} | {name} | {url}")
        print(f"  → valid={result.valid}, confidence={result.confidence}")
    except Exception as e:
        print(f"{city} | {url} | ERROR: {e}")

In [ ]:
# helper: append single row to CSV in S3
def append_result(city, name, url, valid, confidence, error=""):
    # read existing CSV (or start fresh)
    try:
        body = s3.get_object(Bucket=BUCKET, Key=OUTPUT_KEY)["Body"].read().decode("utf-8")
        existing = body
    except s3.exceptions.NoSuchKey:
        existing = "city,name,url,valid,confidence,error\n"

    output = io.StringIO(existing)
    output.seek(0, io.SEEK_END)
    writer = csv.writer(output)
    writer.writerow([city, name, url, valid, confidence, error])
    s3.put_object(Bucket=BUCKET, Key=OUTPUT_KEY, Body=output.getvalue().encode("utf-8"))

In [ ]:
# run all remaining URLs with incremental save
import time

for i, (city, name, url) in enumerate(remaining):
    country = city_country.get(city, "")
    try:
        result = await classify(city, country, url)
        append_result(city, name, url, result.valid, result.confidence)
        print(f"[{i+1}/{len(remaining)}] {city} | {url[:60]} → valid={result.valid}, conf={result.confidence}")
    except Exception as e:
        append_result(city, name, url, "", "", str(e)[:200])
        print(f"[{i+1}/{len(remaining)}] {city} | {url[:60]} → ERROR: {e}")
    time.sleep(0.1)

print("Done!")